In [1]:
import pyabf
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

In [2]:
# Archivos del experimento (número de archivo → intensidad del láser en mA)
laser_ma = {8: 500, 9: 700, 10: 900, 11: 1000, 12: 1100, 13: 1200, 14: 1400}

archivos = [8, 9, 10, 11, 12, 13, 14]
fases = ["control", "laser", "rec"]
labels= ["Control", "Láser", "Recuperación"]
# colores = {"control": "#1a6fad", "laser": "#c0392b", "rec": "#2e7d32"} Algunos los he ido cambiando de tonalidad según la gráfica

ruta_base = r"C:\Users\Usuario\Documents\TFG\Experimento ABRIL 2026\27042026 patch"
ruta_analisis = r"C:\Users\Usuario\Documents\TFG\Experimento ABRIL 2026\ANÁLISIS"

FASE_DURACION = 150  # segundos por fase

## Inspección de un archivo con la librería pyabf

In [3]:
abf = pyabf.ABF(rf"{ruta_base}\26427008.abf")

print("Número de sweeps:",  abf.sweepCount) 
# Los contamos para saber cómo lo entiende Python, ya que en los primeros experimentos teníamos varios sweeps por archivo
print("Canales:", abf.channelCount) # Señal + TTL
print("Frecuencia de muestreo:", abf.dataRate, "Hz")
print("Duración total:",abf.sweepLengthSec, "s")
print("Unidades:", abf.adcUnits)

Número de sweeps: 1
Canales: 2
Frecuencia de muestreo: 5000 Hz
Duración total: 480.0 s
Unidades: ['mV', 'V']


## Detección de tiempos de transición TTL

El TTL marca cuándo se activó y desactivó el láser en cada registro (teóricamente, pudo ser un poco impreciso).
Detectamos los flancos de subida y bajada para obtener los límites exactos de cada fase.

In [4]:
print(f"{'Archivo':>8} | {'Inicio láser (s)':>17} | {'Fin láser (s)':>14} | {'Duración total (s)':>18}")
print("-" * 68)

for num in archivos:
    abf = pyabf.ABF(rf"{ruta_base}\264270{num:02d}.abf")
    abf.setSweep(0, channel=1)
    tiempo = abf.sweepX
    ttl = abf.sweepY

    laser_on = ttl > 2.5
    diff = np.diff(laser_on.astype(int))
    subida= np.where(diff ==  1)[0]
    bajada = np.where(diff == -1)[0]

    t_inicio = tiempo[subida[0]]
    t_fin = tiempo[bajada[0]]
    t_total  = tiempo[-1]

    print(f"{num:>8} | {t_inicio:>17.1f} | {t_fin:>14.1f} | {t_total:>18.1f}")

 Archivo |  Inicio láser (s) |  Fin láser (s) | Duración total (s)
--------------------------------------------------------------------
       8 |             163.1 |          314.8 |              480.0
       9 |             154.3 |          310.3 |              480.0
      10 |             155.6 |          311.4 |              480.0
      11 |             152.1 |          304.9 |              480.0
      12 |             156.4 |          314.4 |              480.0
      13 |             153.7 |          314.3 |              480.0
      14 |             152.9 |          307.8 |              480.0


## Diccionario con los tiempos de transición

Estos valores los usaré como referencia en todo el análisis posterior. Cogeré 150 s en cada fase.

In [5]:
archivos_info = {
    8:  {"control": (0,163.1), "laser": (163.1, 314.8), "rec": (314.8, 480.0)},
    9:  {"control": (0,154.3), "laser": (154.3, 310.3), "rec": (310.3, 480.0)},
    10: {"control": (0,155.6), "laser": (155.6, 311.4), "rec": (311.4, 480.0)},
    11: {"control": (0,152.1), "laser": (152.1, 304.9), "rec": (304.9, 480.0)},
    12: {"control": (0,156.4), "laser": (156.4, 314.4), "rec": (314.4, 480.0)},
    13: {"control": (0,153.7), "laser": (153.7, 314.3), "rec": (314.3, 480.0)},
    14: {"control": (0,152.9), "laser": (152.9, 307.8), "rec": (307.8, 480.0)},
}

## Detección de pulsos de corriente

Usamos la mediana del voltaje de cada fase como estimación del potencial de reposo.
La mediana es robusta frente a los picos de los APs, ya que la neurona pasa ~70% del tiempo en reposo.
El umbral de detección es reposo + 10 mV.

Si dos flancos de subida consecutivos están separados menos de 2 s, se consideran el mismo pulso
(criterio de refractariedad para evitar que un AP dentro del pulso genere una detección extra).

In [7]:
print(f"{'Archivo':>8} | {'Fase':>8} | {'Reposo (mV)':>12} | {'Umbral (mV)':>12} | {'Pulsos detectados':>18}")
print("-" * 70)

for num in archivos:
    abf = pyabf.ABF(rf"{ruta_base}\264270{num:02d}.abf")
    abf.setSweep(0, channel=0)
    tiempo = abf.sweepX
    vm = abf.sweepY
    fs = abf.dataRate

    for fase in fases:
        t0 = archivos_info[num][fase][0]
        t1 = t0 + FASE_DURACION             # Aquí ya cogemos solamente los 150s primeros de cada fase 
        mask = (tiempo >= t0) & (tiempo <= t1)
        t_fase  = tiempo[mask]
        vm_fase = vm[mask]

        reposo = np.median(vm_fase)
        umbral = reposo + 10

        por_encima = (vm_fase > umbral).astype(int)
        flancos_subida = np.where(np.diff(por_encima) == 1)[0]
        flancos_bajada = np.where(np.diff(por_encima) == -1)[0]

        pulsos = []
        i = 0
        while i < len(flancos_subida):
            t_ini = t_fase[flancos_subida[i]]
            bajadas_validas = flancos_bajada[flancos_bajada > flancos_subida[i]]
            if len(bajadas_validas) > 0:
                pulsos.append((t_ini, t_fase[bajadas_validas[0]]))
            i += 1
            while i < len(flancos_subida) and t_fase[flancos_subida[i]] - t_ini < 2.0:
                i += 1

        print(f"{num:>8} | {fase:>8} | {reposo:>12.1f} | {umbral:>12.1f} | {len(pulsos):>18}")

 Archivo |     Fase |  Reposo (mV) |  Umbral (mV) |  Pulsos detectados
----------------------------------------------------------------------
       8 |  control |        -72.8 |        -62.8 |                 45
       8 |    laser |        -71.8 |        -61.8 |                 45
       8 |      rec |        -72.1 |        -62.1 |                 44
       9 |  control |        -72.4 |        -62.4 |                 44
       9 |    laser |        -71.5 |        -61.5 |                 45
       9 |      rec |        -71.2 |        -61.2 |                 45
      10 |  control |        -70.9 |        -60.9 |                 45
      10 |    laser |        -69.8 |        -59.8 |                 45
      10 |      rec |        -70.5 |        -60.5 |                 45
      11 |  control |        -68.8 |        -58.8 |                 45
      11 |    laser |        -68.9 |        -58.9 |                 44
      11 |      rec |        -70.8 |        -60.8 |                 45
      

## Detección de potenciales de acción y conteo por pulso --> Métrica de actividad promedio

Usamos find_peaks con umbral en 0 mV y distancia mínima de 50 ms entre picos.
El umbral de 0 mV garantiza que solo se detecten despolarizaciones completas.
La distancia mínima evita dobles detecciones pero permite capturar ráfagas de hasta 4 APs por pulso.

In [9]:
print(f"{'Archivo':>8} | {'Fase':>8} | {'APs totales':>12} | {'Pulsos':>7} | {'Con AP':>7} | {'Media APs/pulso':>16}")
print("-" * 75)

resultados = []

for num in archivos:
    abf = pyabf.ABF(rf"{ruta_base}\264270{num:02d}.abf")
    abf.setSweep(0, channel=0)
    tiempo= abf.sweepX
    vm= abf.sweepY
    fs= abf.dataRate

    for fase in fases:
        t0 = archivos_info[num][fase][0]
        t1 = t0 + FASE_DURACION
        mask = (tiempo >= t0) & (tiempo <= t1)
        t_fase = tiempo[mask]
        vm_fase = vm[mask]

        # Pulsos
        reposo = np.median(vm_fase)
        umbral = reposo + 10
        por_encima = (vm_fase > umbral).astype(int)
        flancos_subida = np.where(np.diff(por_encima) ==  1)[0]
        flancos_bajada = np.where(np.diff(por_encima) == -1)[0]

        pulsos = []
        i = 0
        while i < len(flancos_subida):
            t_ini = t_fase[flancos_subida[i]]
            bajadas_validas = flancos_bajada[flancos_bajada > flancos_subida[i]]
            if len(bajadas_validas) > 0:
                pulsos.append((t_ini, t_fase[bajadas_validas[0]]))
            i += 1
            while i < len(flancos_subida) and t_fase[flancos_subida[i]] - t_ini < 2.0:
                i += 1

        # APs
        picos, _ = find_peaks(vm_fase, height=0, distance=int(0.05 * fs))
        n_aps= len(picos)

        # APs por pulso (margen de 0.5 s para capturar el último AP del pulso)
        aps_por_pulso = [
            int(np.sum((t_fase[picos] >= t_ini) & (t_fase[picos] <= t_fin + 0.5)))
            for t_ini, t_fin in pulsos
        ]

        media= round(np.mean(aps_por_pulso), 2) if aps_por_pulso else 0
        pulsos_con_ap = int(np.sum(np.array(aps_por_pulso) > 0))

        print(f"{num:>8} | {fase:>8} | {n_aps:>12} | {len(pulsos):>7} | {pulsos_con_ap:>7} | {media:>16}")

        resultados.append({
            "archivo":num,
            "laser_mA": laser_ma[num],
            "fase":fase,
            "n_aps":n_aps,
            "n_pulsos":len(pulsos),
            "pulsos_con_ap": pulsos_con_ap,
            "media_aps":media,
        })

 Archivo |     Fase |  APs totales |  Pulsos |  Con AP |  Media APs/pulso
---------------------------------------------------------------------------
       8 |  control |          143 |      45 |      45 |             3.18
       8 |    laser |          160 |      45 |      45 |             3.56
       8 |      rec |           99 |      44 |      44 |             2.16
       9 |  control |           48 |      44 |      44 |             1.07
       9 |    laser |           16 |      45 |      15 |             0.36
       9 |      rec |           79 |      45 |      45 |             1.76
      10 |  control |           89 |      45 |      45 |             1.98
      10 |    laser |           62 |      45 |      45 |             1.38
      10 |      rec |           91 |      45 |      45 |             2.02
      11 |  control |          101 |      45 |      45 |             2.24
      11 |    laser |           50 |      44 |      43 |             1.14
      11 |      rec |           88 |

In [10]:
# Guardo los resultados en un DataFrame para usarlos en los siguientes notebooks
df = pd.DataFrame(resultados)
df

,archivo,laser_mA,fase,n_aps,n_pulsos,pulsos_con_ap,media_aps
0,8,500,control,143,45,45,3.18
1,8,500,laser,160,45,45,3.56
2,8,500,rec,99,44,44,2.16
3,9,700,control,48,44,44,1.07
4,9,700,laser,16,45,15,0.36
5,9,700,rec,79,45,45,1.76
6,10,900,control,89,45,45,1.98
7,10,900,laser,62,45,45,1.38
8,10,900,rec,91,45,45,2.02
9,11,1000,control,101,45,45,2.24
